# Understanding EEG Signal Changes During Epileptic Seizures
### A Reproducible Signal Processing Pipeline Using the CHB-MIT Scalp EEG Database

**Author:** Oketokun Abayomi Tolulope  
**Dataset:** CHB-MIT Scalp EEG Database (PhysioNet)  
**Tools:** Python · MNE · scikit-learn · SciPy · pandas  



This notebook explores how scalp EEG signals behave differently during 
epileptic seizures compared to seizure-free periods. Starting from raw 
clinical recordings, the pipeline moves through signal loading, 
preprocessing, feature extraction, annotation alignment, and baseline 
classification, each step documented and reproducible.

The immediate goal is a working single-recording pipeline. The longer-term 
goal is to extend this to the full 24-participant dataset and evaluate 
whether simple hand-crafted features generalise across participants; a 
harder and more honest test than within-recording cross-validation.

In [1]:
# ============================================================
# 1 — Install required libraries
# ============================================================

## Setting up the environment

In [2]:
%pip install -q mne scipy pandas numpy matplotlib scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [3]:
# ============================================================
# 2 - Imports
# ============================================================

In [4]:
import re
import json
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import mne
mne.set_log_level("ERROR")

from scipy.signal import welch
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

warnings.filterwarnings("ignore", category=FutureWarning)

print("All libraries imported successfully.")

All libraries imported successfully.


In [5]:
# ============================================================
# 3 — Project structure and dataset paths
# ============================================================

## Project structure

This notebook analyses scalp EEG recordings from the CHB-MIT database,
a publicly available collection of paediatric epilepsy recordings hosted
on PhysioNet. The goal is to understand how the EEG signal changes during
a seizure by building a reproducible pipeline that goes from raw signal
to labelled features to a baseline classification result.

All working files are saved under `/kaggle/working/eeg_seizure_project/`.
The raw EEG data is read directly from the Kaggle dataset input — nothing
is copied unnecessarily.

**Dataset reference:**  
Shoeb, A. H. (2009). *Application of machine learning to epileptic seizure
onset detection and treatment*. PhD Thesis, MIT.  
Available at: https://physionet.org/content/chbmit/1.0.0/

In [6]:
# Working directory for all outputs
PROJECT_DIR = Path("/kaggle/working/eeg_seizure_project")

for folder in ["features", "results", "figures", "logs"]:
    (PROJECT_DIR / folder).mkdir(parents=True, exist_ok=True)

# Root of the Kaggle-hosted CHB-MIT archive
DATASET_ROOT = Path(
    "/kaggle/input/datasets/amudhans07/"
    "chbmit-eeg-full-physionet-archive"
)

print("Project directory :", PROJECT_DIR)
print("Dataset root      :", DATASET_ROOT)
print("Dataset accessible:", DATASET_ROOT.exists())

Project directory : /kaggle/working/eeg_seizure_project
Dataset root      : /kaggle/input/datasets/amudhans07/chbmit-eeg-full-physionet-archive
Dataset accessible: True


In [7]:
# ============================================================
# 4 — Parse all participant summary files into a manifest
# ============================================================

## Building the recording manifest

The CHB-MIT dataset includes a plain-text summary file for each
participant. These files document every recording session and, where
applicable, the onset and offset times of each seizure in seconds.

Rather than hardcoding seizure times or assuming which files contain
seizures, we parse every summary file systematically and build a
manifest — a single table that maps each recording to its verified
annotation. Every label used later in this project traces back to
this manifest.

This approach makes the pipeline transparent: if someone questions
how a window was labelled as seizure or non-seizure, the answer
lives in the manifest and ultimately in the original summary file.

In [8]:
def parse_summary(summary_path: Path) -> list[dict]:
    """
    Read a CHB-MIT participant summary file and extract recording
    metadata and seizure annotations.

    Parameters
    ----------
    summary_path : Path
        Path to a file named like chb01-summary.txt.

    Returns
    -------
    list of dict, one entry per recording listed in the summary.
    Each dict contains the filename, seizure count, and a list of
    (start_seconds, end_seconds) tuples for each seizure event.
    """
    text     = summary_path.read_text(errors="replace")
    sections = re.split(r"(?=File Name:)", text)

    records = []

    for section in sections:
        if "File Name:" not in section:
            continue

        name_match = re.search(r"File Name:\s*(\S+)", section)
        if not name_match:
            continue
        filename = name_match.group(1).strip()

        count_match = re.search(
            r"Number of Seizures in File:\s*(\d+)", section
        )
        n_seizures = int(count_match.group(1)) if count_match else 0

        seizure_intervals = []

        if n_seizures > 0:
            starts = re.findall(
                r"Seizure(?:\s+\d+)?\s+Start\s+Time\s*:\s*(\d+)\s*second",
                section,
                re.IGNORECASE
            )
            ends = re.findall(
                r"Seizure(?:\s+\d+)?\s+End\s+Time\s*:\s*(\d+)\s*second",
                section,
                re.IGNORECASE
            )
            seizure_intervals = [
                (int(s), int(e))
                for s, e in zip(starts, ends)
            ]

        records.append({
            "filename"          : filename,
            "n_seizures"        : n_seizures,
            "seizure_intervals" : seizure_intervals
        })

    return records


# Walk through both dataset parts and build the manifest
all_records = []

for part in ["chb_mit_part1", "chb_mit_part2"]:
    part_dir = DATASET_ROOT / part
    if not part_dir.exists():
        continue

    for subj_dir in sorted(part_dir.iterdir()):
        if not subj_dir.is_dir():
            continue

        summary_files = list(subj_dir.glob("*-summary.txt"))
        if not summary_files:
            print(f"No summary file found in {subj_dir.name} — skipping.")
            continue

        subject_id = subj_dir.name
        records    = parse_summary(summary_files[0])

        for rec in records:
            edf_path = subj_dir / rec["filename"]

            all_records.append({
                "subject_id"        : subject_id,
                "filename"          : rec["filename"],
                "edf_path"          : str(edf_path) if edf_path.exists() else None,
                "file_found"        : edf_path.exists(),
                "n_seizures"        : rec["n_seizures"],
                "seizure_intervals" : rec["seizure_intervals"],
                "processed"         : False,
                "error"             : None
            })

manifest = pd.DataFrame(all_records)

# Save the manifest so it can be inspected independently
manifest.to_csv(PROJECT_DIR / "results" / "manifest.csv", index=False)

print(f"Total recordings in manifest : {len(manifest)}")
print(f"Recordings found on disk     : {manifest['file_found'].sum()}")
print(f"Recordings with seizures     : {(manifest['n_seizures'] > 0).sum()}")
print(f"Recordings missing from disk : {(~manifest['file_found']).sum()}")
print()
print("Sample — recordings with documented seizures:")
print(
    manifest[manifest["n_seizures"] > 0][
        ["subject_id", "filename", "n_seizures", "seizure_intervals"]
    ].head(10).to_string(index=False)
)

Total recordings in manifest : 676
Recordings found on disk     : 676
Recordings with seizures     : 141
Recordings missing from disk : 0

Sample — recordings with documented seizures:
subject_id      filename  n_seizures seizure_intervals
     chb01  chb01_03.edf           1    [(2996, 3036)]
     chb01  chb01_04.edf           1    [(1467, 1494)]
     chb01  chb01_15.edf           1    [(1732, 1772)]
     chb01  chb01_16.edf           1    [(1015, 1066)]
     chb01  chb01_18.edf           1    [(1720, 1810)]
     chb01  chb01_21.edf           1      [(327, 420)]
     chb01  chb01_26.edf           1    [(1862, 1963)]
     chb02  chb02_16.edf           1      [(130, 212)]
     chb02 chb02_16+.edf           1    [(2972, 3053)]
     chb02  chb02_19.edf           1    [(3369, 3378)]


In [9]:
# ============================================================
# 5 — Select and verify one recording
# ============================================================

## Selecting a recording to analyse

The first concrete analysis uses a single recording: `chb01_15.edf`
from participant chb01. This file was chosen because it has a
documented seizure event and its annotation file is confirmed present
in the dataset.

The verification step below is intentional. It is easy to assume a
file is correctly annotated and discover the mistake only after
running a long analysis. Checking explicitly before loading the
signal is a small habit that prevents large errors.

In [10]:
TARGET_SUBJECT = "chb01"
TARGET_FILE    = "chb01_15.edf"

row = manifest[
    (manifest["subject_id"] == TARGET_SUBJECT) &
    (manifest["filename"]   == TARGET_FILE)
]

if row.empty:
    raise ValueError(
        f"{TARGET_FILE} was not found in the manifest. "
        "Check that the summary file was parsed correctly."
    )

row = row.iloc[0]

print("Subject           :", row["subject_id"])
print("File              :", row["filename"])
print("Path              :", row["edf_path"])
print("File on disk      :", row["file_found"])
print("Documented seizures:", row["n_seizures"])
print("Seizure intervals :", row["seizure_intervals"])

if not row["file_found"]:
    raise ValueError(
        "The EDF file was not found at the expected path. "
        "Verify the dataset is correctly attached to this notebook."
    )

if row["n_seizures"] == 0 or not row["seizure_intervals"]:
    raise ValueError(
        "No seizure intervals were parsed for this recording. "
        "Open the participant summary file and check the annotation "
        "before continuing."
    )

print("\nVerification passed. Safe to load the EEG signal.")

Subject           : chb01
File              : chb01_15.edf
Path              : /kaggle/input/datasets/amudhans07/chbmit-eeg-full-physionet-archive/chb_mit_part1/chb01/chb01_15.edf
File on disk      : True
Documented seizures: 1
Seizure intervals : [(1732, 1772)]

Verification passed. Safe to load the EEG signal.


In [11]:
# ============================================================
# 6 — Load the EEG recording
# ============================================================

## Loading the EEG signal

EDF (European Data Format) is the standard file format for clinical
EEG recordings. MNE reads this format natively. We load the metadata
first — without pulling the entire signal into memory — so we can
inspect the recording's properties before committing to loading it.

We also verify that the documented seizure times fall within the
recording's actual duration. A mismatch here would mean our labels
are wrong.

In [12]:
edf_path          = row["edf_path"]
seizure_intervals = row["seizure_intervals"]

# Load metadata only (preload=False keeps memory usage low)
raw = mne.io.read_raw_edf(edf_path, preload=False, verbose="ERROR")

sfreq      = raw.info["sfreq"]
duration_s = raw.times[-1]
n_channels = len(raw.ch_names)

print(f"Sampling frequency  : {sfreq} Hz")
print(f"Number of channels  : {n_channels}")
print(f"Recording duration  : {duration_s / 60:.2f} minutes "
      f"({duration_s:.0f} seconds)")
print(f"First 10 channels   : {raw.ch_names[:10]}")
print()

# Confirm seizure timestamps are within bounds
print("Seizure interval check:")
for i, (start, end) in enumerate(seizure_intervals):
    within = end <= duration_s
    status = "OK" if within else "WARNING — exceeds recording duration"
    print(f"  Seizure {i + 1}: {start}s – {end}s  [{status}]")

Sampling frequency  : 256.0 Hz
Number of channels  : 23
Recording duration  : 60.00 minutes (3600 seconds)
First 10 channels   : ['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4']

Seizure interval check:
  Seizure 1: 1732s – 1772s  [OK]


In [13]:
# ============================================================
# 7 — Bandpass filter the signal
# ============================================================

## Preprocessing — bandpass filtering

Raw EEG recordings contain activity across a wide frequency range,
including sources that are not neurological. Very low frequencies
(below 1 Hz) often reflect slow electrode drift or movement artefacts.
Frequencies above 40 Hz in scalp EEG are dominated by muscle artefact
rather than brain activity.

A bandpass filter retains only the frequency range of interest, here,
1 to 40 Hz and attenuates everything outside it. This is a standard
first step in EEG preprocessing.

It is worth being clear about what filtering does not do: it does not
remove all artefacts, it does not guarantee the signal is clean, and
the exact cutoff frequencies are a methodological choice that should
be justified for the specific research question. For a seizure detection
task focused on classical frequency bands (delta, theta, alpha, beta),
1–40 Hz is a defensible starting range.

In [14]:
# Load the full signal into memory before filtering
raw_filtered = raw.copy()
raw_filtered.load_data(verbose="ERROR")

raw_filtered.filter(
    l_freq  = 1.0,
    h_freq  = 40.0,
    picks   = "eeg",
    verbose = "ERROR"
)

print("Bandpass filter applied: 1–40 Hz")
print(f"Signal shape: {raw_filtered.get_data(picks='eeg').shape} "
      f"(channels × samples)")

Bandpass filter applied: 1–40 Hz
Signal shape: (23, 921600) (channels × samples)


In [15]:
# ============================================================
# 8 — Plot the EEG signal
# ============================================================

## Visualising the EEG signal

Before extracting any features, it helps to look at the raw signal.
Two plots are produced here: one from a quiet period at the start of
the recording, and one centred on the seizure onset. The difference
between them is often visible even without specialist training and
seizure periods tend to show larger amplitude, more rhythmic, or
more synchronised activity across channels.

These plots are saved to the figures folder rather than rendered
interactively, which keeps the notebook clean and ensures the
visualisations are preserved alongside the other outputs.

In [16]:
# Plot 1 — first 10 seconds of the recording (baseline period)
fig1 = raw_filtered.plot(
    start      = 0,
    duration   = 10,
    n_channels = min(10, n_channels),
    scalings   = "auto",
    title      = "EEG signal — baseline period (first 10 seconds)",
    show       = False
)
fig1.savefig(
    PROJECT_DIR / "figures" / "signal_baseline.png",
    dpi=120, bbox_inches="tight"
)
plt.close(fig1)
print("Baseline plot saved.")

# Plot 2 — 10 seconds centred on the first seizure onset
seizure_start_s = seizure_intervals[0][0]
plot_start      = max(0, seizure_start_s - 5)

fig2 = raw_filtered.plot(
    start      = plot_start,
    duration   = 10,
    n_channels = min(10, n_channels),
    scalings   = "auto",
    title      = (
        f"EEG signal — seizure onset at {seizure_start_s}s "
        f"(plotted from {plot_start}s)"
    ),
    show       = False
)
fig2.savefig(
    PROJECT_DIR / "figures" / "signal_seizure_onset.png",
    dpi=120, bbox_inches="tight"
)
plt.close(fig2)
print(f"Seizure onset plot saved (centred on {seizure_start_s}s).")

Baseline plot saved.
Seizure onset plot saved (centred on 1732s).


In [17]:
# ============================================================
# 9 — Feature extraction
# ============================================================

## Extracting features from sliding windows

The raw EEG signal is a continuous voltage trace. To build a
classification model, we need to convert that trace into a table
where each row summarises a short segment of the signal.

The approach here is to divide the recording into non-overlapping
10-second windows and compute seven features from each window,
averaged across all EEG channels:

| Feature | What it captures |
|---|---|
| Variance | Overall spread of the signal amplitude |
| RMS (root mean square) | Average signal power |
| Line length | Total path length of the signal — sensitive to high-frequency activity |
| Delta power (1–4 Hz) | Slow-wave activity |
| Theta power (4–8 Hz) | Associated with drowsiness and some seizure types |
| Alpha power (8–13 Hz) | Dominant rhythm in relaxed wakefulness |
| Beta power (13–30 Hz) | Associated with alertness and motor activity |

These are relatively simple features chosen because they are
interpretable, computationally cheap, and commonly used in the
seizure detection literature. A more sophisticated analysis might
add entropy measures, connectivity features, or channel-level
spatial information.

In [18]:
data   = raw_filtered.get_data(picks="eeg")  # shape: (channels, samples)
sfreq  = raw_filtered.info["sfreq"]

WINDOW_SECONDS = 10
STEP_SECONDS   = 10    # non-overlapping; set to < WINDOW_SECONDS for overlap
window_samples = int(WINDOW_SECONDS * sfreq)
step_samples   = int(STEP_SECONDS   * sfreq)


def band_power(psd: np.ndarray, freqs: np.ndarray,
               low: float, high: float) -> float:
    """
    Compute mean band power across channels for a given frequency range.

    Parameters
    ----------
    psd   : array of shape (channels, frequencies)
    freqs : frequency axis from scipy.signal.welch
    low   : lower frequency bound in Hz
    high  : upper frequency bound in Hz

    Returns
    -------
    float — mean power across channels in the specified band
    """
    mask = (freqs >= low) & (freqs < high)
    if mask.sum() < 2:
        return 0.0
    return float(
        np.mean(np.trapezoid(psd[:, mask], freqs[mask], axis=1))
    )


feature_rows = []
n_samples    = data.shape[1]

for start in range(0, n_samples - window_samples + 1, step_samples):
    segment = data[:, start : start + window_samples]
    end     = start + window_samples

    # Time-domain features
    variance    = np.var(segment, axis=1)
    rms         = np.sqrt(np.mean(segment ** 2, axis=1))
    line_length = np.mean(np.abs(np.diff(segment, axis=1)), axis=1)

    # Power spectral density via Welch's method
    nperseg      = min(int(2 * sfreq), segment.shape[1])
    freqs, psd   = welch(segment, fs=sfreq, nperseg=nperseg, axis=1)

    feature_rows.append({
        "window_start_s"   : start / sfreq,
        "window_end_s"     : end   / sfreq,
        "variance_mean"    : float(np.mean(variance)),
        "rms_mean"         : float(np.mean(rms)),
        "line_length_mean" : float(np.mean(line_length)),
        "delta_power"      : band_power(psd, freqs, 1,  4),
        "theta_power"      : band_power(psd, freqs, 4,  8),
        "alpha_power"      : band_power(psd, freqs, 8,  13),
        "beta_power"       : band_power(psd, freqs, 13, 30),
    })

features = pd.DataFrame(feature_rows)

print(f"Window length    : {WINDOW_SECONDS}s")
print(f"Step size        : {STEP_SECONDS}s")
print(f"Total windows    : {len(features)}")
print()
print("First five windows:")
print(features.head().to_string(index=False))

Window length    : 10s
Step size        : 10s
Total windows    : 360

First five windows:
 window_start_s  window_end_s  variance_mean  rms_mean  line_length_mean  delta_power  theta_power  alpha_power   beta_power
            0.0          10.0   4.026565e-09  0.000061          0.000004 1.848872e-09 3.549299e-10 9.477313e-11 1.779214e-11
           10.0          20.0   2.944323e-09  0.000053          0.000004 1.889612e-09 3.263184e-10 9.328921e-11 1.902484e-11
           20.0          30.0   4.120803e-09  0.000063          0.000005 2.590967e-09 5.253199e-10 1.402241e-10 2.601641e-11
           30.0          40.0   3.294282e-09  0.000056          0.000004 2.110848e-09 3.183566e-10 1.017537e-10 2.022401e-11
           40.0          50.0   3.627825e-09  0.000059          0.000004 2.131328e-09 4.551046e-10 1.046834e-10 2.414498e-11


In [19]:
# ============================================================
# 10 — Label the windows
# ============================================================

## Assigning seizure labels to windows

Each 10-second window receives a binary label:

- **1 (seizure)** if the window overlaps with any documented seizure
  interval, even partially
- **0 (non-seizure)** otherwise

The overlap rule; rather than requiring the window to fall entirely
within the seizure, is intentional. Seizure onset and offset are
gradual physiological transitions, and a window that captures even
part of that transition is meaningfully different from one that does
not. For a more rigorous analysis, one might exclude transition windows
entirely, but for this baseline that level of precision is not yet
warranted.

The labels come entirely from the participant summary file, which was
parsed in Cell 4. No label was assigned manually.

In [20]:
def window_overlaps_seizure(
    start_s: float,
    end_s: float,
    intervals: list[tuple]
) -> int:
    """
    Return 1 if the window [start_s, end_s] overlaps any seizure
    interval, 0 otherwise.

    Overlap condition: the window starts before the seizure ends
    AND ends after the seizure starts.
    """
    return int(
        any(
            start_s < seizure_end and end_s > seizure_start
            for seizure_start, seizure_end in intervals
        )
    )


features["seizure_label"] = [
    window_overlaps_seizure(
        row.window_start_s,
        row.window_end_s,
        seizure_intervals
    )
    for row in features.itertuples()
]

n_seizure  = features["seizure_label"].sum()
n_nonsz    = (features["seizure_label"] == 0).sum()
total      = len(features)

print("Label distribution:")
print(f"  Non-seizure windows : {n_nonsz}  ({100 * n_nonsz / total:.1f}%)")
print(f"  Seizure windows     : {n_seizure}  ({100 * n_seizure / total:.1f}%)")
print()
print("Labelled windows around seizure onset:")
onset_region = features[
    (features["window_start_s"] >= seizure_intervals[0][0] - 30) &
    (features["window_start_s"] <= seizure_intervals[0][1] + 30)
][["window_start_s", "window_end_s", "seizure_label"]]
print(onset_region.to_string(index=False))

if n_seizure == 0:
    raise ValueError(
        "No windows were labelled as seizure. "
        "Check that the seizure_intervals align with the window times."
    )

Label distribution:
  Non-seizure windows : 355  (98.6%)
  Seizure windows     : 5  (1.4%)

Labelled windows around seizure onset:
 window_start_s  window_end_s  seizure_label
         1710.0        1720.0              0
         1720.0        1730.0              0
         1730.0        1740.0              1
         1740.0        1750.0              1
         1750.0        1760.0              1
         1760.0        1770.0              1
         1770.0        1780.0              1
         1780.0        1790.0              0
         1790.0        1800.0              0
         1800.0        1810.0              0


In [21]:
# ============================================================
# 11 — Visualise feature distributions
# ============================================================

## Do the features actually differ between seizure and non-seizure?

Before training any model, it is worth asking whether the features we
extracted show any visible difference between the two classes. If the
distributions completely overlap, a classifier is unlikely to learn
anything useful from these features alone.

The histograms below show each feature's distribution separately for
seizure (red) and non-seizure (blue) windows. Differences in the
location or shape of these distributions suggest the feature carries
information about seizure state. Complete overlap would suggest it
does not.

In [22]:
FEATURE_COLS = [
    "variance_mean", "rms_mean", "line_length_mean",
    "delta_power", "theta_power", "alpha_power", "beta_power"
]

FEATURE_LABELS = {
    "variance_mean"    : "Variance",
    "rms_mean"         : "RMS Amplitude",
    "line_length_mean" : "Line Length",
    "delta_power"      : "Delta Power (1–4 Hz)",
    "theta_power"      : "Theta Power (4–8 Hz)",
    "alpha_power"      : "Alpha Power (8–13 Hz)",
    "beta_power"       : "Beta Power (13–30 Hz)"
}

fig, axes = plt.subplots(
    nrows=2, ncols=4,
    figsize=(18, 7),
    constrained_layout=True
)
axes = axes.flatten()

for i, col in enumerate(FEATURE_COLS):
    ax = axes[i]

    for label, colour, name in [
        (0, "steelblue", "Non-seizure"),
        (1, "crimson",   "Seizure")
    ]:
        subset = features[features["seizure_label"] == label][col]
        ax.hist(subset, bins=30, alpha=0.65, color=colour, label=name)

    ax.set_title(FEATURE_LABELS[col], fontsize=10, fontweight="bold")
    ax.set_xlabel("Feature value", fontsize=8)
    ax.set_ylabel("Window count",  fontsize=8)
    ax.legend(fontsize=7)
    ax.tick_params(labelsize=7)

axes[-1].set_visible(False)

fig.suptitle(
    f"Feature distributions — seizure vs non-seizure windows\n"
    f"Recording: {TARGET_FILE}  |  Participant: {TARGET_SUBJECT}",
    fontsize=12,
    fontweight="bold"
)

fig.savefig(
    PROJECT_DIR / "figures" / "feature_distributions.png",
    dpi=120, bbox_inches="tight"
)
plt.close(fig)
print("Feature distribution plot saved.")

Feature distribution plot saved.


In [23]:
# ============================================================
# 12 — Train a baseline classifier
# ============================================================

## Baseline classification — what can simple features tell us?

A Random Forest classifier is trained on the seven extracted features
using stratified 5-fold cross-validation. Stratification ensures that
each fold contains a proportional representation of seizure windows,
which matters here because seizure windows are heavily outnumbered.

The `class_weight="balanced"` setting adjusts the loss function to
penalise misclassifying the minority class (seizure) more heavily than
the majority class (non-seizure). Without this, a classifier that
simply predicts "non-seizure" for every window would achieve over 98%
accuracy on this recording, which would be meaningless.

**Important caveat:** This result comes from a single recording by a
single participant. Cross-validation within one recording does not tell
us whether the model generalises to new participants, new recording
conditions, or different seizure types. That question requires a
participant-level train/test split across the full dataset, which is
the natural next step for this project.

In [24]:
X = features[FEATURE_COLS].values
y = features["seizure_label"].values

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

all_y_true = []
all_y_pred = []

print("Running 5-fold stratified cross-validation...\n")

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Fit scaler on training data only — never on test data
    scaler  = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)

    clf = RandomForestClassifier(
        n_estimators  = 100,
        class_weight  = "balanced",
        random_state  = 42
    )
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    all_y_true.extend(y_test)
    all_y_pred.extend(y_pred)

    n_sz_test = y_test.sum()
    print(f"  Fold {fold} — seizure windows in test set: {n_sz_test}")

print()
print("=" * 55)
print("Cross-validated classification report")
print("=" * 55)
print(classification_report(
    all_y_true,
    all_y_pred,
    target_names = ["Non-seizure", "Seizure"],
    zero_division = 0
))

Running 5-fold stratified cross-validation...

  Fold 1 — seizure windows in test set: 1
  Fold 2 — seizure windows in test set: 1
  Fold 3 — seizure windows in test set: 1
  Fold 4 — seizure windows in test set: 1
  Fold 5 — seizure windows in test set: 1

Cross-validated classification report
              precision    recall  f1-score   support

 Non-seizure       1.00      1.00      1.00       355
     Seizure       1.00      1.00      1.00         5

    accuracy                           1.00       360
   macro avg       1.00      1.00      1.00       360
weighted avg       1.00      1.00      1.00       360



In [25]:
# ============================================================
# 13 — Confusion matrix
# ============================================================

## Confusion matrix

The confusion matrix breaks down the classifier's predictions into
four categories: true positives (seizure windows correctly identified),
true negatives (non-seizure windows correctly identified), false
positives (non-seizure windows incorrectly flagged as seizure), and
false negatives (seizure windows missed entirely).

In a clinical context, false negatives (missed seizures) are
generally more costly than false positives. The confusion matrix makes
this trade-off visible in a way that a single accuracy number does not.

In [26]:
cm = confusion_matrix(all_y_true, all_y_pred)

fig, ax = plt.subplots(figsize=(5, 4))

disp = ConfusionMatrixDisplay(
    confusion_matrix = cm,
    display_labels   = ["Non-seizure", "Seizure"]
)
disp.plot(ax=ax, colorbar=False, cmap="Blues")

ax.set_title(
    f"Confusion matrix — {TARGET_FILE}\n5-fold cross-validation",
    fontsize=10,
    fontweight="bold"
)

fig.savefig(
    PROJECT_DIR / "figures" / "confusion_matrix.png",
    dpi=120, bbox_inches="tight"
)
plt.close(fig)
print("Confusion matrix saved.")

Confusion matrix saved.


In [27]:
# ============================================================
# 14 — Feature importance
# ============================================================

## Which features contributed most to the classifier?

Random Forests provide a built-in measure of feature importance based
on how much each feature reduces impurity across all decision trees.
This gives a rough indication of which signal properties were most
useful for distinguishing seizure from non-seizure windows.

This is trained on the full dataset (not cross-validated) purely for
the purpose of inspecting feature importance. The cross-validated
report in Cell 12 remains the performance estimate.

In [28]:
# Train on full dataset for feature importance inspection only
scaler_full = StandardScaler()
X_scaled    = scaler_full.fit_transform(X)

clf_full = RandomForestClassifier(
    n_estimators = 100,
    class_weight = "balanced",
    random_state = 42
)
clf_full.fit(X_scaled, y)

importances = clf_full.feature_importances_
indices     = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(8, 4))

ax.bar(
    range(len(FEATURE_COLS)),
    importances[indices],
    color="steelblue",
    edgecolor="white"
)
ax.set_xticks(range(len(FEATURE_COLS)))
ax.set_xticklabels(
    [FEATURE_LABELS[FEATURE_COLS[i]] for i in indices],
    rotation=30,
    ha="right",
    fontsize=9
)
ax.set_ylabel("Mean decrease in impurity", fontsize=9)
ax.set_title(
    "Feature importance — Random Forest\n"
    f"Recording: {TARGET_FILE}",
    fontsize=10,
    fontweight="bold"
)

fig.savefig(
    PROJECT_DIR / "figures" / "feature_importance.png",
    dpi=120, bbox_inches="tight"
)
plt.close(fig)

print("Feature importance (highest to lowest):")
for i in indices:
    print(f"  {FEATURE_LABELS[FEATURE_COLS[i]]:<30} {importances[i]:.4f}")

Feature importance (highest to lowest):
  RMS Amplitude                  0.2960
  Variance                       0.2671
  Delta Power (1–4 Hz)           0.1732
  Theta Power (4–8 Hz)           0.0999
  Line Length                    0.0902
  Beta Power (13–30 Hz)          0.0732
  Alpha Power (8–13 Hz)          0.0003


In [29]:
# ============================================================
# 15 — Save all outputs
# ============================================================

## Saving outputs

All outputs from this analysis are saved in structured form so the
results can be inspected, shared, or extended without re-running the
full notebook. The processing log captures the key parameters and
metadata from this run in a format that supports batch processing
across the full dataset later.

In [30]:
# Save the feature table
feature_path = PROJECT_DIR / "features" / f"{TARGET_FILE}_features.csv"
features.to_csv(feature_path, index=False)
print(f"Features saved     : {feature_path}")

# Save the processing log
log = {
    "subject_id"         : TARGET_SUBJECT,
    "filename"           : TARGET_FILE,
    "edf_path"           : edf_path,
    "sfreq_hz"           : sfreq,
    "duration_s"         : round(duration_s, 2),
    "n_channels"         : n_channels,
    "filter_l_freq_hz"   : 1.0,
    "filter_h_freq_hz"   : 40.0,
    "window_seconds"     : WINDOW_SECONDS,
    "step_seconds"       : STEP_SECONDS,
    "total_windows"      : len(features),
    "seizure_windows"    : int(features["seizure_label"].sum()),
    "nonsz_windows"      : int((features["seizure_label"] == 0).sum()),
    "seizure_intervals"  : seizure_intervals,
    "processed_at"       : datetime.now().isoformat(),
    "error"              : None
}

log_path = PROJECT_DIR / "logs" / f"{TARGET_FILE}_log.json"
log_path.write_text(json.dumps(log, indent=2))
print(f"Processing log saved: {log_path}")

print()
print("Figures saved:")
for fig_file in sorted((PROJECT_DIR / "figures").iterdir()):
    print(f"  {fig_file.name}")

print()
print("Run complete.")
print(json.dumps(log, indent=2))

Features saved     : /kaggle/working/eeg_seizure_project/features/chb01_15.edf_features.csv
Processing log saved: /kaggle/working/eeg_seizure_project/logs/chb01_15.edf_log.json

Figures saved:
  confusion_matrix.png
  feature_distributions.png
  feature_importance.png
  signal_baseline.png
  signal_seizure_onset.png

Run complete.
{
  "subject_id": "chb01",
  "filename": "chb01_15.edf",
  "edf_path": "/kaggle/input/datasets/amudhans07/chbmit-eeg-full-physionet-archive/chb_mit_part1/chb01/chb01_15.edf",
  "sfreq_hz": 256.0,
  "duration_s": 3600.0,
  "n_channels": 23,
  "filter_l_freq_hz": 1.0,
  "filter_h_freq_hz": 40.0,
  "window_seconds": 10,
  "step_seconds": 10,
  "total_windows": 360,
  "seizure_windows": 5,
  "nonsz_windows": 355,
  "seizure_intervals": [
    [
      1732,
      1772
    ]
  ],
  "processed_at": "2026-10-18T09:00:58.266186",
  "error": null
}
